In [1]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

# convert section to lowercase because they are inconsistent across both datasets
events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)


In [2]:
print("\nColumns:")
print(events.column_names)
print("\nFeatures:")
print(events.features)
print(f"\nTotal rows: {len(events)}")

import random
from pprint import pprint

n_samples = 1

indices = random.sample(range(len(events)), n_samples)

print(f"\n{n_samples} Random events:")
for idx in indices:
    sample = events[idx]
    print(f"\nSample[{idx}]:")
    pprint(sample)

# count articles in each section
from collections import Counter

section_counts = Counter(events["section"])

print("\nSection counts:")
for section, count in section_counts.most_common():
    print(f"\t{section}: {count}")



Columns:
['month', 'year', 'day', 'section', 'content', 'instruction', 'response']

Features:
{'month': Value('string'), 'year': Value('int64'), 'day': Value('int64'), 'section': Value('string'), 'content': Value('string'), 'instruction': Value('string'), 'response': Value('string')}

Total rows: 10712

1 Random events:

Sample[8103]:
{'content': 'Israeli settler violence\n'
            'Israeli settlers kill two Palestinians near Ramallah in the West '
            'Bank. One of the victims, identified as a Palestinian-American, '
            'was beaten to death whilst the other victim was shot in the '
            'chest. (Reuters)',
 'day': 11,
 'instruction': 'How did the July 11, 2025 incident near Ramallah escalate '
                'into lethal Israeli settler violence that left two '
                'Palestinians dead, including a Palestinian-American '
                'reportedly beaten to death?',
 'month': 'July',
 'response': 'The lethal Israeli settler violence that occur

In [4]:
import requests, time

ollama_base_url = "http://localhost:11434/api/"
embedding_models = [
    "mxbai-embed-large",
    "nomic-embed-text"
]

for embedding_model in embedding_models:

    start = time.time()
    print(f"\n**** Embeddings using model {embedding_model}")
    response = requests.post(
        ollama_base_url + "embeddings",
        json={
            "model": embedding_model,
            "prompt": sample["instruction"]
        }
    )
    end = time.time()
    print(f"{len(response.json()["embedding"])} dimensions generated in {round(end-start, 3)}s")


**** Embeddings using model mxbai-embed-large
1024 dimensions generated in 0.049s

**** Embeddings using model nomic-embed-text
768 dimensions generated in 0.639s


In [5]:
llm_models = [
    "qwen3.5:9b",
    "llama3.2:3b",
    "llama3.2:1b",
    "gpt-oss:20b"
]


for llm_model in llm_models:
    start = time.time()

    print(f"\n\n*** Trial generate with {llm_model}:")
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Summarize the following text clearly and concisely:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Summary: " + response.json()["response"])
    end = time.time()
    print(f"Summary Time: {round(end - start)}s") 

    start = time.time()
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Give three main concise and clear points from the text:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Main points:\n" + response.json()["response"])

    end = time.time()
    print(f"Main points time: {round(end - start)}s") 



*** Trial generate with qwen3.5:9b:
Summary: Israeli settlers killed two Palestinians, including one Palestinian-American, near Ramallah in the West Bank; one was beaten to death and the other was shot.
Summary Time: 24s
Main points:
1. Israeli settlers killed two Palestinians near Ramallah in the West Bank.
2. One victim, identified as a Palestinian-American, was beaten to death.
3. The other victim was shot in the chest.
Main points time: 25s


*** Trial generate with llama3.2:3b:
Summary: Two Palestinians were killed by Israeli settlers in the West Bank: one was beaten to death and the other was shot in the chest.
Summary Time: 0s
Main points:
Here are three concise and clear points from the text:

1. Two Palestinians were killed by Israeli settlers in the West Bank.
2. The victims were attacked near Ramallah: one was beaten to death and the other was shot in the chest.
3. One of the victims, identified as a Palestinian-American, has been named as the victim who was beaten to deat